# Phase 2: Hybrid Weapon Detection Training (v5 — DagsHub S3 Download)

**Architecture**: Google Drive (code + weights) × DagsHub S3 Storage (41k+ image dataset)

| Component | Location |
|-----------|----------|
| Source Code & Weights | Google Drive (persistent across sessions) |
| Dataset (images/labels) | DagsHub Storage (`dagshub-drive` S3 bucket) → downloaded once to Colab SSD |
| Training Checkpoints | Google Drive `models/weights/` |

> **Strategy**: Dataset is downloaded ONCE from DagsHub S3 to `/content/yolo_dataset/` (Colab SSD).
> Training then reads from local disk — fastest possible I/O, no streaming overhead.

## Step 1 — Environment & Dependencies
Install DagsHub, Ultralytics, and force **Numpy < 2.0** to avoid the Ultralytics / Colab T4 binary incompatibility.

In [ ]:
# ── Install packages ────────────────────────────────────────────────────────
%pip install -q dagshub ultralytics albumentations timm

# ── Force Numpy < 2.0 (binary ABI compatibility with Ultralytics on T4) ─────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "numpy"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy<2.0"], check=True)

# ── Verify ───────────────────────────────────────────────────────────────────
import importlib, numpy as np, torch, os, sys
from pathlib import Path

print(f"✅ Numpy  : {np.__version__}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — check Runtime type!'}")

if np.__version__.startswith('2.'):
    print("\n⚠️  [ACTION REQUIRED] Numpy 2.x detected.\n"
          "    Please click 'RESTART SESSION' in the Colab popup, then re-run from Step 2.")

## Step 2 — Configuration

Set your credentials and paths here. **All user-editable values are in one place.**

> ℹ️ Your DagsHub access token can be generated at: [dagshub.com/user/settings/tokens](https://dagshub.com/user/settings/tokens)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║              USER-EDITABLE CONFIGURATION — change these values           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os, sys
from pathlib import Path

# ── DagsHub credentials ───────────────────────────────────────────────────
DAGSHUB_USERNAME    = "TAMZIRT-MOHAMED"         # ← your DagsHub username
DAGSHUB_TOKEN       = "7a8e6bf40efcf2040ecc529e0bda4412b08e3a62" # ← paste token OR leave blank to use env var

#   Get token: dagshub.com/user/settings/tokens
#   Alternatively, set environment variable: DAGSHUB_USER_TOKEN

# ── Storage repo (where your S3 dataset lives) ────────────────────────────
DAGSHUB_STORAGE_REPO = "dagshub-drive"          # ← the DagsHub Storage repo name
#   Confirmed by URL: dagshub.com/TAMZIRT-MOHAMED/dagshub-drive

# ── Path inside DagsHub Storage ───────────────────────────────────────────
STORAGE_DATASET_PATH = "data/processed/yolo_dataset"   # ← path inside S3 bucket

# ── Local Colab destination (SSD, fast I/O for training) ─────────────────
LOCAL_DATASET_PATH   = "/content/yolo_dataset"  # where dataset will be downloaded

# ── Google Drive path to your source code ────────────────────────────────
GDRIVE_PROJECT_PATH  = "/content/drive/MyDrive/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System-v2/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System/"

# ── Resolve token ────────────────────────────────────────────────────────
if not DAGSHUB_TOKEN:
    DAGSHUB_TOKEN = os.environ.get("DAGSHUB_USER_TOKEN", "")
if not DAGSHUB_TOKEN:
    import getpass
    DAGSHUB_TOKEN = getpass.getpass("🔑 Paste your DagsHub access token: ")

print("Configuration loaded:")
print(f"  Storage repo   : https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_STORAGE_REPO}")
print(f"  S3 dataset path: {STORAGE_DATASET_PATH}")
print(f"  Local dest     : {LOCAL_DATASET_PATH}")
print(f"  Token          : {'✅ set (' + str(len(DAGSHUB_TOKEN)) + ' chars)' if DAGSHUB_TOKEN else '❌ MISSING'}")

## Step 2b — Storage Diagnostic (Run Once to Find Correct Path)

This cell probes every known DagsHub S3 endpoint variant and the DagsHub REST API.
**Run this before Step 2c** to identify the correct configuration for your account.
You can skip this once your setup is confirmed working.

In [ ]:
"""                                                                         
DagsHub Storage Diagnostic                                                  
──────────────────────────────────────────────────────────────────────────  
Tries every known access method and prints exactly what works.              
The ✅ lines tell you which endpoint+bucket combination to use in Step 2c.  
"""

import requests, boto3, botocore, os

U   = DAGSHUB_USERNAME       # from config cell
R   = DAGSHUB_STORAGE_REPO   # "dagshub-drive"
TOK = DAGSHUB_TOKEN

SEP = "=" * 65
print(SEP)
print(" DAGSHUB STORAGE DIAGNOSTIC")
print(SEP)

# ─────────────────────────────────────────────────────────────────────────
# METHOD 1 — DagsHub REST API (most reliable, no S3 quirks)
# ─────────────────────────────────────────────────────────────────────────
print("\n[1] DagsHub REST API — list storage root")

def dagshub_api_ls(owner, repo, path="", token=TOK):
    """List files in DagsHub Storage via REST API."""
    url = f"https://dagshub.com/api/v1/repos/{owner}/{repo}/storage"
    params = {"path": path or "/"}
    resp = requests.get(url, auth=(owner, token), params=params, timeout=15)
    return resp

resp = dagshub_api_ls(U, R)
print(f"  Status : {resp.status_code}")
if resp.ok:
    try:
        j = resp.json()
        print("  ✅ REST API works! Top-level entries:")
        entries = j if isinstance(j, list) else j.get('entries', j.get('files', [j]))
        for e in entries[:20]:
            name = e.get('path') or e.get('name') or str(e)
            print(f"     {name}")
    except Exception as ex:
        print(f"  ✅ Got 200 but unexpected format: {resp.text[:300]}")
else:
    print(f"  ❌ {resp.text[:200]}")

# Also try listing the specific dataset path via REST
print("\n[1b] DagsHub REST API — list data/processed/yolo_dataset")
resp2 = dagshub_api_ls(U, R, path="data/processed/yolo_dataset")
print(f"  Status : {resp2.status_code}")
if resp2.ok:
    print("  ✅ Dataset path found via REST API!")
    try:
        j = resp2.json()
        entries = j if isinstance(j, list) else j.get('entries', j.get('files', []))
        for e in list(entries)[:10]:
            print(f"     {e.get('path') or e.get('name') or e}")
    except:
        print(f"  Response: {resp2.text[:300]}")
else:
    print(f"  ❌ {resp2.text[:200]}")

# ─────────────────────────────────────────────────────────────────────────
# METHOD 2 — boto3 Variant A
# endpoint = https://dagshub.com   |   Bucket = "username/repo"
# ─────────────────────────────────────────────────────────────────────────
print("\n[2A] boto3 — endpoint=https://dagshub.com  |  Bucket=username/repo")
try:
    s3a = boto3.client(
        "s3",
        endpoint_url="https://dagshub.com",
        aws_access_key_id=U,
        aws_secret_access_key=TOK,
        config=botocore.config.Config(
            signature_version="s3v4",
            s3={"addressing_style": "path"},
        ),
    )
    bucket_a = f"{U}/{R}"
    resp_a = s3a.list_objects_v2(Bucket=bucket_a, Prefix="", MaxKeys=10)
    keys = [o["Key"] for o in resp_a.get("Contents", [])]
    print(f"  ✅ Works! Bucket='{bucket_a}'  Top keys: {keys}")
except Exception as e:
    print(f"  ❌ {type(e).__name__}: {e}")

# ─────────────────────────────────────────────────────────────────────────
# METHOD 2B — boto3 Variant B
# endpoint = https://dagshub.com/{user}/{repo}.s3   |   Bucket = repo name
# ─────────────────────────────────────────────────────────────────────────
print(f"\n[2B] boto3 — endpoint=https://dagshub.com/{U}/{R}.s3  |  Bucket={R}")
try:
    s3b = boto3.client(
        "s3",
        endpoint_url=f"https://dagshub.com/{U}/{R}.s3",
        aws_access_key_id=U,
        aws_secret_access_key=TOK,
        config=botocore.config.Config(
            signature_version="s3v4",
            s3={"addressing_style": "path"},
        ),
    )
    resp_b = s3b.list_objects_v2(Bucket=R, Prefix="", MaxKeys=10)
    keys = [o["Key"] for o in resp_b.get("Contents", [])]
    print(f"  ✅ Works! Bucket='{R}'  Top keys: {keys}")
except Exception as e:
    print(f"  ❌ {type(e).__name__}: {e}")

# ─────────────────────────────────────────────────────────────────────────
# METHOD 3 — DagsHub Python library (dagshub.upload / RepoAPI)
# ─────────────────────────────────────────────────────────────────────────
print("\n[3] dagshub Python library — explore available APIs")
try:
    import dagshub
    # Set credentials in env so dagshub library picks them up
    os.environ["DAGSHUB_USER_TOKEN"] = TOK
    os.environ["DAGSHUB_USERNAME"]   = U

    # Try RepoAPI
    try:
        from dagshub.common.api.repo_api import RepoAPI
        api = RepoAPI(owner=U, name=R)
        print(f"  RepoAPI instantiated: {api}")
        # List available methods
        methods = [m for m in dir(api) if not m.startswith('_') and 'stor' in m.lower()]
        print(f"  Storage-related methods: {methods}")
        for method in methods[:3]:
            try:
                result = getattr(api, method)()
                print(f"    {method}() → {result}")
            except Exception as me:
                print(f"    {method}() → {type(me).__name__}: {me}")
    except ImportError as ie:
        print(f"  RepoAPI not available: {ie}")

    # Try dagshub.upload Repo
    try:
        from dagshub.upload import Repo as UploadRepo
        repo_ul = UploadRepo(owner=U, name=R)
        print(f"  UploadRepo available: {type(repo_ul)}")
        methods = [m for m in dir(repo_ul) if not m.startswith('_')]
        print(f"  Methods: {methods[:15]}")
    except Exception as ue:
        print(f"  UploadRepo: {type(ue).__name__}: {ue}")

except Exception as e:
    print(f"  ❌ dagshub library error: {e}")

print("\n" + SEP)
print(" SUMMARY")
print(SEP)
print("Look at which methods above showed ✅")
print("Share the full output — we'll configure the download cell from this.")

In [ ]:
import boto3
import botocore
import torch
import numpy as np
from google.colab import drive
from tqdm import tqdm

# ── 1. Mount Google Drive (source code + weight checkpoints) ─────────────
drive.mount('/content/drive')

PROJECT_ROOT = Path(GDRIVE_PROJECT_PATH)
if PROJECT_ROOT.exists():
    os.chdir(str(PROJECT_ROOT))
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print(f"✅ GDrive project root aligned: {PROJECT_ROOT}")
else:
    raise FileNotFoundError(
        f"❌ Project root not found at:\n   {PROJECT_ROOT}\n"
        "   Check GDRIVE_PROJECT_PATH in the config cell."
    )

from models.hybrid_model import HybridWeaponDetector
print("✅ HybridWeaponDetector imported.")

# ── 2. Download dataset from DagsHub S3 Storage to Colab SSD ─────────────
#    DagsHub Storage exposes an S3-compatible endpoint:
#    https://dagshub.com/{username}/{repo}.s3
#    Credentials: username = access key, token = secret key

S3_ENDPOINT = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_STORAGE_REPO}.s3"
S3_BUCKET   = DAGSHUB_STORAGE_REPO   # bucket name = repo name
S3_PREFIX   = STORAGE_DATASET_PATH.rstrip("/") + "/"

s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=DAGSHUB_USERNAME,
    aws_secret_access_key=DAGSHUB_TOKEN,
    config=botocore.config.Config(signature_version="s3v4"),
)

os.makedirs(LOCAL_DATASET_PATH, exist_ok=True)

print(f"\n🚀 Syncing dataset from DagsHub S3 → {LOCAL_DATASET_PATH}")
print(f"   Endpoint : {S3_ENDPOINT}")
print(f"   Prefix   : {S3_BUCKET}/{S3_PREFIX}")
print("   (Only downloads files not already present — safe to re-run)\n")

paginator = s3.get_paginator("list_objects_v2")
pages = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX)

all_objects = []
for page in pages:
    all_objects.extend(page.get("Contents", []))

if not all_objects:
    raise RuntimeError(
        f"❌ No objects found at s3://{S3_BUCKET}/{S3_PREFIX}\n"
        "   Check STORAGE_DATASET_PATH and your token permissions."
    )

print(f"Found {len(all_objects):,} objects to sync.")

skipped = 0
downloaded = 0
for obj in tqdm(all_objects, desc="Downloading"):
    key = obj["Key"]
    # Compute local path by stripping the S3 prefix and prepending local root
    relative_path = key[len(S3_PREFIX):]
    if not relative_path:   # skip the directory object itself
        continue
    local_file = Path(LOCAL_DATASET_PATH) / relative_path
    local_file.parent.mkdir(parents=True, exist_ok=True)
    # Skip if file already exists with same size (resume-safe)
    if local_file.exists() and local_file.stat().st_size == obj["Size"]:
        skipped += 1
        continue
    s3.download_file(S3_BUCKET, key, str(local_file))
    downloaded += 1

print(f"\n✅ Sync complete: {downloaded:,} downloaded, {skipped:,} already present.")

# Set DATA_YAML_PATH for subsequent cells
DATA_YAML_PATH = Path(LOCAL_DATASET_PATH) / "data.yaml"
if DATA_YAML_PATH.exists():
    print(f"✅ data.yaml confirmed at: {DATA_YAML_PATH}")
else:
    print(f"⚠️  data.yaml not found at {DATA_YAML_PATH} — check STORAGE_DATASET_PATH")

## Step 3 — Verify Dataset
Confirm `data.yaml` is present and the image counts look right before starting training.

In [ ]:
# DATA_YAML_PATH is already set by the download cell above.
# This cell prints a summary and reads a preview of data.yaml.

import yaml

if not DATA_YAML_PATH.exists():
    raise FileNotFoundError(f"data.yaml not found at {DATA_YAML_PATH}. Re-run Step 2.")

with open(DATA_YAML_PATH) as f:
    cfg = yaml.safe_load(f)

print("=" * 50)
print("data.yaml contents:")
print(f"  nc     : {cfg.get('nc')}")
print(f"  names  : {cfg.get('names')}")
print(f"  train  : {cfg.get('train')}")
print(f"  val    : {cfg.get('val')}")
print("=" * 50)

# Count images
dataset_root = DATA_YAML_PATH.parent
for split in ['train', 'val']:
    split_path = dataset_root / split / 'images'
    if split_path.exists():
        n = len(list(split_path.iterdir()))
        print(f"✅ {split:5s} images: {n:,}")
    else:
        print(f"⚠️  {split} images dir not found at {split_path}")

## Step 4 — Hybrid Trainer (Two-Phase Schedule)

**Phase 1** (frozen backbone, 10 epochs): Trains neck + head only → stabilises feature alignment.

**Phase 2** (full fine-tune, remaining epochs): Unfreeze backbone → global optimisation.

Class weight `[1.0, 1.0, 2.5]` applies **2.5× focal penalty on the Confuser class** to suppress false positives.

In [ ]:
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm

# ── DataLoader factory ───────────────────────────────────────────────────────
def get_dataloaders(data_yaml_path: str, batch_size: int = 16, imgsz: int = 640):
    """Build train/val DataLoaders using Ultralytics YOLODataset."""
    data_cfg = check_det_dataset(data_yaml_path)
    train_set = YOLODataset(
        img_path=data_cfg['train'], imgsz=imgsz,
        augment=True, batch_size=batch_size, task='detect', data=data_cfg
    )
    val_set = YOLODataset(
        img_path=data_cfg['val'], imgsz=imgsz,
        augment=False, batch_size=batch_size, task='detect', data=data_cfg
    )
    train_loader = DataLoader(
        train_set, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, collate_fn=train_set.collate_fn
    )
    val_loader = DataLoader(
        val_set, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True, collate_fn=val_set.collate_fn
    )
    print(f"✅ DataLoaders ready — {len(train_set):,} train / {len(val_set):,} val samples.")
    return train_loader, val_loader


# ── Two-phase Trainer ────────────────────────────────────────────────────────
class HybridTrainer:
    """
    Two-phase training schedule:
      Phase 1 — frozen backbone, warm up neck + head.
      Phase 2 — full fine-tune with unfrozen backbone.

    Checkpoints are saved to `weights_dir` on Google Drive for persistence.
    """

    def __init__(
        self,
        model,
        train_loader,
        val_loader,
        device: str = "cuda",
        weights_dir: str = "models/weights",
        freeze_epochs: int = 10,
    ):
        self.device        = device
        self.model         = model.to(device)
        self.train_loader  = train_loader
        self.val_loader    = val_loader
        self.scaler        = GradScaler()
        self.weights_dir   = Path(weights_dir)
        self.freeze_epochs = freeze_epochs
        self.best_loss     = float('inf')
        self.weights_dir.mkdir(parents=True, exist_ok=True)
        self.criterion     = model.head.compute_loss

    # ── Backbone freeze / unfreeze ───────────────────────────────────────────
    def _set_backbone_frozen(self, frozen: bool):
        for param in self.model.backbone.parameters():
            param.requires_grad = not frozen
        state = "frozen  ❄️" if frozen else "unfrozen 🔥"
        print(f"  Backbone {state}")

    def _build_optimizer(self):
        """Create optimiser over currently trainable parameters only."""
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        return optim.AdamW(trainable, lr=1e-4, weight_decay=1e-4)

    # ── Checkpoint helpers ───────────────────────────────────────────────────
    def save_checkpoint(self, optimizer, epoch: int, is_best: bool = False, tag: str = None):
        state = {
            'epoch':               epoch,
            'model_state_dict':    self.model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_loss':           self.best_loss,
        }
        torch.save(state, self.weights_dir / "last.pt")
        if is_best:
            torch.save(self.model.state_dict(), self.weights_dir / "best.pt")
            print(f"  💾 best.pt updated (loss={self.best_loss:.4f})")
        if tag:
            torch.save(state, self.weights_dir / f"epoch_{tag}.pt")

    def load_checkpoint(self, optimizer):
        """Resume from last.pt if it exists."""
        ckpt_path = self.weights_dir / "last.pt"
        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location=self.device)
            self.model.load_state_dict(ckpt['model_state_dict'])
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            self.best_loss = ckpt.get('best_loss', float('inf'))
            start_epoch = ckpt['epoch'] + 1
            print(f"✅ Resumed from epoch {ckpt['epoch']} — best_loss={self.best_loss:.4f}")
            return start_epoch
        return 1

    # ── Single epoch ────────────────────────────────────────────────────────
    def train_epoch(self, optimizer, epoch: int, total_epochs: int) -> float:
        self.model.train()
        total_loss = 0.0
        phase_tag  = "Phase 1 (frozen backbone)" if epoch <= self.freeze_epochs else "Phase 2 (full fine-tune)"
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}/{total_epochs} [{phase_tag}]")
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            optimizer.zero_grad()
            with autocast():
                preds = self.model(imgs)
                loss  = self.criterion(preds, batch, self.device)
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        return total_loss / len(self.train_loader)

    # ── Main loop ────────────────────────────────────────────────────────────
    def run(self, total_epochs: int = 50, resume: bool = True):
        """
        Run two-phase training.
        - Epochs 1 … freeze_epochs : backbone frozen
        - Epochs freeze_epochs+1 … total_epochs : backbone unfrozen
        """
        # Phase 1 setup
        self._set_backbone_frozen(True)
        optimizer  = self._build_optimizer()
        start_epoch = 1

        if resume:
            start_epoch = self.load_checkpoint(optimizer)

        phase2_started = (start_epoch > self.freeze_epochs)
        if phase2_started:
            print("Resuming in Phase 2 — unfreezing backbone.")
            self._set_backbone_frozen(False)
            optimizer = self._build_optimizer()   # rebuild with all params
            self.load_checkpoint(optimizer)       # reload optimizer state

        for epoch in range(start_epoch, total_epochs + 1):

            # Phase transition
            if epoch == self.freeze_epochs + 1 and not phase2_started:
                print("\n── Phase 2: Unfreezing backbone for full fine-tune ──")
                self._set_backbone_frozen(False)
                optimizer = self._build_optimizer()   # fresh optimiser over all params
                phase2_started = True

            avg_loss = self.train_epoch(optimizer, epoch, total_epochs)
            is_best  = avg_loss < self.best_loss
            if is_best:
                self.best_loss = avg_loss

            # Permanent checkpoint every 5 epochs
            tag = str(epoch) if epoch % 5 == 0 else None
            self.save_checkpoint(optimizer, epoch, is_best=is_best, tag=tag)

            print(f"Epoch {epoch:3d}/{total_epochs} | avg_loss={avg_loss:.4f} | best={self.best_loss:.4f}")

        print("\n🏁 Training complete. Checkpoints saved to:", self.weights_dir)

## Step 5 — Initialise Model & Launch Training

In [ ]:
# ── Device ───────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("⚠️  No GPU detected — training will be very slow. Check Runtime > Change runtime type.")

# ── Model ────────────────────────────────────────────────────────────────────
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", nc=3, device=device)

# ── Red Alert class weight: 2.5× focal penalty on Confuser (class 2) ────────
model.head.alpha = torch.tensor([1.0, 1.0, 2.5], device=device)
print("✅ Class weights set: Weapon=1.0 | Person=1.0 | Confuser=2.5")

# ── DataLoaders ──────────────────────────────────────────────────────────────
train_loader, val_loader = get_dataloaders(
    str(DATA_YAML_PATH),
    batch_size=16,
    imgsz=640,
)

# ── Trainer ──────────────────────────────────────────────────────────────────
WEIGHTS_DIR = PROJECT_ROOT / "models" / "weights"
trainer = HybridTrainer(
    model        = model,
    train_loader = train_loader,
    val_loader   = val_loader,
    device       = device,
    weights_dir  = str(WEIGHTS_DIR),
    freeze_epochs= 10,        # Phase 1: 10 frozen epochs
)

# ── Launch ───────────────────────────────────────────────────────────────────
#   resume=True  → automatically continues from last.pt if it exists
#   resume=False → fresh run (ignores existing checkpoints)
trainer.run(total_epochs=50, resume=True)